# Word2Vec: CBOW vs Skip-Gram

Word2Vec trains a neural network to predict words from context, producing dense vector representations where semantically similar words cluster together. Two architectures exist: CBOW predicts a target word from surrounding context words, while Skip-Gram does the reverse. Both produce word embeddings but differ in training dynamics and performance characteristics — CBOW is faster and suits frequent words; Skip-Gram handles rare words better and generally produces richer representations on large corpora.


## Install and Import

In [ ]:
!pip install gensim -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 50.6 MB/s eta 0:00:00


## Text Dataset

A small corpus covering animals, royalty, geography and nature. In production, replace this with Wikipedia, news archives, or domain-specific text — larger and more varied corpora produce significantly better embeddings.


In [ ]:
import re
from gensim.models import Word2Vec

print("Ready.")


Ready.


In [ ]:
raw_text = """
The modern world is shaped by a complex interaction of technology, society, and human behavior. Over the past few decades, digital transformation has significantly altered how people communicate, work, and access information. The rise of the internet and mobile devices has created a highly connected global environment where knowledge can be shared instantly across borders. Social media platforms, in particular, have changed the dynamics of communication, allowing individuals to express ideas, build communities, and influence public opinion. However, this rapid exchange of information has also led to challenges such as misinformation, reduced privacy, and increased dependence on digital systems.

In parallel, scientific advancements continue to push the boundaries of what is possible. Innovations in fields such as biotechnology, space exploration, and renewable energy are opening new opportunities for solving global problems. Researchers are developing sustainable energy solutions to reduce dependence on fossil fuels, while space missions aim to explore distant planets and understand the origins of the universe. These developments not only expand human knowledge but also contribute to economic growth and technological progress.
"""

print(f"Corpus loaded.")


Corpus loaded.


## Preprocessing

Lowercase the text, strip punctuation, and split into sentences. Each sentence becomes a list of tokens — the format gensim expects.


In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    sentences = []
    for line in text.strip().split('\n'):
        words = line.strip().split()
        if len(words) > 1:
            sentences.append(words)
    return sentences

sentences = preprocess(raw_text)
all_words = [w for s in sentences for w in s]

print(f"Sentences : {len(sentences)}")
print(f"Tokens    : {len(all_words)}")
print(f"Unique    : {len(set(all_words))}")
print(f"\nSample:")
for s in sentences[:3]:
    print(f"  {' '.join(s)}")


Sentences : 2
Tokens    : 174
Unique    : 131

Sample:
  the modern world is shaped by a complex interaction of technology society and human behavior over the past few decades digital transformation has significantly altered how people communicate work and access information the rise of the internet and mobile devices has created a highly connected global environment where knowledge can be shared instantly across borders social media platforms in particular have changed the dynamics of communication allowing individuals to express ideas build communities and influence public opinion however this rapid exchange of information has also led to challenges such as misinformation reduced privacy and increased dependence on digital systems
  in parallel scientific advancements continue to push the boundaries of what is possible innovations in fields such as biotechnology space exploration and renewable energy are opening new opportunities for solving global problems researchers are developing

## Train Both Models

Key parameters: `vector_size` sets embedding dimensions; `window` controls context span; `min_count` ignores rare tokens; `sg=0` selects CBOW, `sg=1` selects Skip-Gram; `epochs` controls training passes.


In [ ]:
cbow_model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=0,        # CBOW
    epochs=200,
    seed=42
)

sg_model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=3,
    min_count=1,
    sg=1,        # Skip-Gram
    epochs=200,
    seed=42
)

print("Both models trained.")


Both models trained.


## Vocabulary Size

Both models share the same vocabulary since they train on the same corpus. Every unique token gets a 50-dimensional vector.


In [ ]:
vocab = list(cbow_model.wv.key_to_index.keys())

print(f"Vocabulary size: {len(vocab)} words")
print(f"\nAll tokens:")
print(sorted(vocab))


Vocabulary size: 131 words

All tokens:
['a', 'access', 'across', 'advancements', 'aim', 'allowing', 'also', 'altered', 'and', 'are', 'as', 'be', 'behavior', 'biotechnology', 'borders', 'boundaries', 'build', 'but', 'by', 'can', 'challenges', 'changed', 'communicate', 'communication', 'communities', 'complex', 'connected', 'continue', 'contribute', 'created', 'decades', 'dependence', 'developing', 'developments', 'devices', 'digital', 'distant', 'dynamics', 'economic', 'energy', 'environment', 'exchange', 'expand', 'exploration', 'explore', 'express', 'few', 'fields', 'for', 'fossil', 'fuels', 'global', 'growth', 'has', 'have', 'highly', 'how', 'however', 'human', 'ideas', 'in', 'increased', 'individuals', 'influence', 'information', 'innovations', 'instantly', 'interaction', 'internet', 'is', 'knowledge', 'led', 'media', 'misinformation', 'missions', 'mobile', 'modern', 'new', 'not', 'of', 'on', 'only', 'opening', 'opinion', 'opportunities', 'origins', 'over', 'parallel', 'particular'

## Word Vectors

Each word is encoded as a list of 50 floats. The absolute values are not interpretable individually — what matters is the geometric relationship between vectors. Only the first 10 values are shown here.


In [ ]:
a = ['technology', 'society', 'human', 'information', 'internet']

print(f"{'Word':<10}  {'Model':<10}  Vector (first 10 values)")
print("-" * 72)

for b in a:
    for label, model in [("CBOW", cbow_model), ("Skip-Gram", sg_model)]:
        vec = model.wv[b]
        vals = ', '.join([f'{v:.3f}' for v in vec[:10]])
        print(f"{b:<10}  {label:<10}  [{vals} ...]")
    print()

Word        Model       Vector (first 10 values)
------------------------------------------------------------------------
technology  CBOW        [-0.021, -0.148, -0.185, 0.057, -0.206, 0.095, -0.129, 0.266, -0.204, 0.012 ...]
technology  Skip-Gram   [-0.050, -0.123, -0.230, 0.051, -0.229, 0.136, -0.140, 0.227, -0.147, -0.006 ...]

society     CBOW        [-0.058, -0.145, -0.164, 0.060, -0.204, 0.079, -0.153, 0.254, -0.180, -0.003 ...]
society     Skip-Gram   [-0.097, -0.126, -0.225, 0.056, -0.242, 0.118, -0.180, 0.231, -0.128, -0.018 ...]

human       CBOW        [-0.070, -0.236, -0.236, 0.105, -0.245, 0.104, -0.183, 0.321, -0.260, -0.006 ...]
human       Skip-Gram   [-0.108, -0.214, -0.250, 0.076, -0.233, 0.086, -0.158, 0.241, -0.167, -0.033 ...]

information  CBOW        [-0.069, -0.224, -0.226, 0.105, -0.233, 0.110, -0.178, 0.319, -0.265, 0.037 ...]
information  Skip-Gram   [-0.087, -0.129, -0.201, 0.062, -0.202, 0.132, -0.167, 0.195, -0.187, 0.048 ...]

internet    CBOW        [-0

## Top-5 Similar Words

Similarity is measured by cosine distance between vectors. The two models often agree on the top candidates but differ in ranking, reflecting their different training objectives.


In [ ]:
new_vec = ['technology', 'society', 'human', 'information', 'internet']

for x in new_vec:
    print(f"Query: '{x}'")
    print(f"  {'Rank':<5}  {'CBOW':<28}  {'Skip-Gram':<28}")
    print(f"  {'----':<5}  {'-'*26:<28}  {'-'*26:<28}")

    cbow_sim = cbow_model.wv.most_similar(x, topn=5)
    sg_sim   = sg_model.wv.most_similar(x, topn=5)

    for i, ((cw, cs), (sw, ss)) in enumerate(zip(cbow_sim, sg_sim), 1):
        cb = f"{cw:<14} {cs:.4f}"
        sk = f"{sw:<14} {ss:.4f}"
        print(f"  {i:<5}  {cb:<28}  {sk:<28}")
    print()


Query: 'technology'
  Rank   CBOW                          Skip-Gram                   
  ----   --------------------------    --------------------------  
  1      in             0.9976         society        0.9942       
  2      to             0.9973         complex        0.9900       
  3      has            0.9973         interaction    0.9897       
  4      of             0.9972         behavior       0.9892       
  5      the            0.9971         shaped         0.9879       

Query: 'society'
  Rank   CBOW                          Skip-Gram                   
  ----   --------------------------    --------------------------  
  1      in             0.9968         technology     0.9942       
  2      ideas          0.9967         behavior       0.9916       
  3      knowledge      0.9967         over           0.9903       
  4      human          0.9967         interaction    0.9897       
  5      a              0.9965         complex        0.9863       

Query: 'h

## Summary

**CBOW** averages context embeddings to predict a center word. It trains faster and performs well when the corpus is small or words are frequent.

**Skip-Gram** treats each context word as a separate training example. It takes longer to train but generalises better to rare words and typically produces higher-quality embeddings on large corpora.

Both models represent words as dense vectors in a shared geometric space. Distance and direction in that space encode semantic relationships — synonyms cluster, antonyms tend to oppose, and analogy tasks resolve to simple vector arithmetic.
